# 03 — Caso B: direzione condivisa, energia libera

**Assunzioni:** (1) sorgente unica; (2) direzione del neutrone circa costante (campo lontano);
scattering isotropo in CM. Il parametro condiviso è la direzione $\Omega_n$. L'energia $E_n$ di
ogni evento è una nuisance con prior largo (piatto su 0.5–6 MeV).

Per ogni direzione candidata $\Omega$, l'angolo osservato fra traccia e candidato è
$\hat\theta = \angle(\hat t_k, \Omega)$. La verosimiglianza marginalizza sia $E_n$ sia l'angolo vero
$\theta$:
$$L_k(\Omega) = \int d\theta \int dE_n\, \pi(E_n)\,\frac{\cos\theta}{\pi}\,
N(\hat\theta;\theta,\sigma_\theta)\, N(\hat E_p; E_n\cos^2\theta, \sigma_{E_p}).$$
Il termine $\cos\theta/\pi$ è la densità della traccia per steradiante (notebook 01).
L'approssimazione con errore polare gaussiano 1D è etichettata **(b)**.

**Etichette dei risultati.** (a) fatto consolidato · (b) stima toy con assunzioni esplicite ·
(c) verificato con Monte Carlo · (d) ipotesi da testare.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import trapezoid
from scipy.stats import kstest, norm

from riptide_toy.constants import SEED, SIGMA_EP, SIGMA_THETA, EN_MIN, EN_MAX

plt.rcParams["figure.dpi"] = 110
rng = np.random.default_rng(SEED)
from riptide_toy import forward_model, grids, kinematics, posterior_B, posterior_C, priors, validate

OMEGA_TRUE = kinematics.direction_from_theta_phi(np.array([0.9]), np.array([2.1]))  # versore (1, 3)
theta_grid, phi_grid = grids.sphere_grid()          # 3000 pixel di area uguale
prior_dir = priors.direction_prior(theta_grid, phi_grid)
omega_grid = kinematics.direction_from_theta_phi(theta_grid, phi_grid)


def simulate(n, rng, smear=True):
    En = rng.uniform(1.0, 5.0, n)
    Ep, track = kinematics.sample_recoil_events(rng, En, OMEGA_TRUE[0])
    Ep_hat = rng.normal(Ep, SIGMA_EP)
    return (Ep_hat, kinematics.smear_direction(rng, track, SIGMA_THETA) if smear else track)


def sky(ax, values, title):
    sc = ax.scatter(np.degrees(phi_grid), 90 - np.degrees(theta_grid), c=values, s=6, cmap="viridis")
    ax.plot(np.degrees(2.1), 90 - np.degrees(0.9), "r*", ms=12)
    ax.set_xlabel("phi [gradi]"); ax.set_ylabel("latitudine [gradi]"); ax.set_title(title)
    return sc

## Esempio guidato: cosa dice un evento solo

Si trascurino per un momento le risoluzioni. Fissata una direzione candidata $\Omega$ a un angolo
$\theta$ dalla traccia osservata, la cinematica impone una sola energia possibile,
$E_n^\ast = E_p/\cos^2\theta$. La verosimiglianza diventa (l'integrale su $E_n$ si fa con la delta)
$$L(\Omega) = \int dE_n\,\pi(E_n)\,\frac{\cos\theta}{\pi}\,\delta(E_p - E_n\cos^2\theta)
= \pi(E_n^\ast)\,\frac{\cos\theta}{\pi}\cdot\frac{1}{\cos^2\theta} = \frac{\pi(E_n^\ast)}{\pi\cos\theta}.$$
Il fattore $1/\cos^2\theta$ è lo jacobiano $|\partial E_n/\partial E_p|$. Esempio con $E_p = 1.5$ MeV e
prior piatto su $[0.5, 6]$ MeV:

| $\theta$ | $\cos^2\theta$ | $E_n^\ast$ [MeV] | $L \propto 1/\cos\theta$ (relativo a 40°) |
|---|---|---|---|
| 20° | 0.883 | 1.70 | 0.815 |
| 40° | 0.587 | 2.56 | 1 |
| 55° | 0.329 | 4.56 | 1.336 |
| 70° | 0.117 | 12.8 | 0 (fuori dal prior) |

Il candidato è escluso quando $E_n^\ast > 6$ MeV, cioè per $\cos^2\theta < 1.5/6 = 0.25$, quindi
oltre $\theta = 60^\circ$. Un evento solo dice quindi «$\Omega_n$ sta entro 60° dalla traccia», con una
leggera preferenza per il bordo del cono, non dove sta $\Omega_n$ **(a)**.

La cella qui sotto calcola la stessa verosimiglianza con il codice. Con risoluzioni molto piccole
riproduce i rapporti della tabella; con quelle vere ($\sigma_\theta = 0.08$ rad) il taglio a 60° si
ammorbidisce e il valore a 55° si abbassa, perché parte della gaussiana in $\theta$ cade oltre il taglio.

In [ ]:
en_grid = grids.energy_grid()
angles = np.radians([20.0, 40.0, 55.0, 70.0])
analytic = np.where(np.cos(angles) ** 2 >= 1.5 / EN_MAX, 1 / np.cos(angles), 0.0)
print("analitico      :", np.round(analytic / analytic[1], 3))
for s_th, s_ep in [(0.01, 0.01), (SIGMA_THETA, SIGMA_EP)]:
    ll = forward_model.loglik_marginal_En_theta(np.array([1.5]), angles[None, :], en_grid, s_ep, s_th,
                                                np.log(priors.energy_prior(en_grid)))[0]
    print(f"sigma_theta={s_th:.2f}: ", np.round(np.exp(ll - ll[1]), 3))

## Un evento solo

Un singolo evento non individua una direzione. Vincola $\Omega_n$ a stare, grosso modo,
nell'emisfero attorno alla traccia, con una preferenza per gli angoli compatibili con $\hat E_p$.
La stella rossa indica la direzione vera.

In [ ]:
D1 = simulate(1, rng)
lp1 = posterior_B.single_event_posterior(D1, (theta_grid, phi_grid), prior_dir)[0]
fig, ax = plt.subplots(figsize=(7, 3.5))
plt.colorbar(sky(ax, np.exp(lp1 - lp1.max()), "posterior di singolo evento"), ax=ax)
plt.show()

## Combinare N eventi

Ogni evento aggiunge un vincolo, e il posterior combinato si concentra attorno alla direzione vera.
Il combinato si calcola con `posterior_C.shared_direction_log_posterior`, che somma le
log-verosimiglianze del Caso B contando il prior una sola volta.

In [ ]:
D = simulate(100, rng)
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for ax, n in zip(axes, (3, 10, 100)):
    Dn = (D[0][:n], D[1][:n])
    c = posterior_C.shared_direction_log_posterior(Dn, theta_grid, phi_grid, prior_dir)
    sky(ax, np.exp(c - c.max()), f"combinato, N={n}")
    err = validate.angular_residual(OMEGA_TRUE, omega_grid[np.argmax(c)][None, :])[0]
    print(f"N={n:3d}: errore del MAP = {np.degrees(err):.2f} gradi")
plt.tight_layout(); plt.show()

Qui il MAP è letto sulla griglia grossolana (3000 pixel, passo circa 3.7°): a N grande l'errore
resta di qualche grado perché è limitato dalla discretizzazione, non dai dati. Il raffinamento
locale più sotto toglie questo limite.

## Perché serve la risoluzione angolare nella verosimiglianza

Una prima versione del modello aveva due caratteristiche. Considerava la traccia misurata senza
errore angolare e poneva $L_k(\Omega) = -\infty$ quando $\hat\theta > 90^\circ$. Con questa
verosimiglianza, un candidato $\Omega$ sopravvive solo se tutte le $N$ tracce stanno nel suo
emisfero anteriore: l'insieme ammesso è l'intersezione di $N$ emisferi. La direzione vera
soddisfa sempre il vincolo, ma i **pixel** della griglia vicini a essa no. Al crescere di N
l'insieme ammesso si riduce a pochi pixel e, a seconda della realizzazione e del passo della
griglia, può svuotarsi del tutto. In quel caso il combinato vale $-\infty$ ovunque e `np.argmax`
restituisce in silenzio il pixel 0, cioè il polo nord della griglia, lontano dalla verità **(c)**.
Anche quando qualche pixel sopravvive, la stima è decisa dal bordo dell'intersezione e non dalla
forma della verosimiglianza.

Qui sotto si contano i pixel ammessi, anche usando tracce esatte (senza smearing):

In [ ]:
for n in (10, 30, 100, 300, 1000):
    counts = []
    for _ in range(8):
        _, track = simulate(n, rng, smear=False)
        ang = kinematics.recoil_angle_from_direction(track, omega_grid)   # (n, 3000)
        counts.append(int(np.sum(np.all(ang <= np.pi / 2, axis=0))))
    print(f"N={n:5d}: pixel con tutte le tracce in avanti (8 realizzazioni) = {counts}")

err_pix0 = validate.angular_residual(OMEGA_TRUE, omega_grid[:1])[0]
print(f"distanza del pixel 0 dalla verita': {np.degrees(err_pix0):.2f} gradi")

Il modello attuale risolve il problema su due fronti:

* **risoluzione angolare nella verosimiglianza**: l'angolo vero viene marginalizzato attorno a
  quello osservato, il taglio netto a 90° diventa morbido e il posterior resta finito su tutta la
  sfera;
* **una guardia**: se il combinato è $-\infty$ ovunque, si solleva un errore invece di restituire
  il pixel 0.

Sugli stessi dati di prima (tracce con smearing), il combinato è finito ovunque:

In [ ]:
c = posterior_C.shared_direction_log_posterior(simulate(300, rng), theta_grid, phi_grid, prior_dir)
print(f"pixel finiti: {np.isfinite(c).sum()} / {c.size}")
print(f"errore del MAP a N=300: {np.degrees(validate.angular_residual(OMEGA_TRUE, omega_grid[np.argmax(c)][None, :])[0]):.2f} gradi")

## Contrazione, risoluzione e pull angolare

Con 3000 pixel la griglia ha un passo di circa 3.7°, che a N grande è più largo del posterior. Si
usa quindi il raffinamento locale (`posterior_C.refine_shared_direction`): prima il combinato sulla
griglia grossolana, poi una calotta fine attorno al massimo.

Per ogni N si ripetono M esperimenti e si misurano:

* l'errore angolare del MAP;
* la risoluzione dichiarata dal posterior;
* il pull angolare, cioè errore diviso risoluzione.

Per un ricostruttore calibrato, l'errore scende come $1/\sqrt N$ e il pull ha rms ≈ 1 **(c)**.

In [ ]:
N_VALUES, M = [10, 30, 100, 300], 6
rows = []
for n in N_VALUES:
    for _ in range(M):
        Dn = simulate(n, rng)
        om, cap_lp, cap_th, cap_ph = posterior_C.refine_shared_direction(Dn, theta_grid, phi_grid, prior_dir)
        cap = kinematics.direction_from_theta_phi(cap_th, cap_ph)
        err = validate.angular_residual(OMEGA_TRUE, om)[0]
        res = validate.posterior_angular_resolution(cap_lp[None, :], cap, om)[0]
        rows.append((n, err, res))
rows = np.array(rows)

rms_err, mean_res = [], []
for n in N_VALUES:
    r = rows[rows[:, 0] == n]
    rms_err.append(np.sqrt(np.mean(r[:, 1] ** 2))); mean_res.append(r[:, 2].mean())
    pull = validate.angular_pull(r[:, 1], r[:, 2])
    print(f"N={n:4d}: rms errore {np.degrees(rms_err[-1]):5.2f} gradi | risoluzione media {np.degrees(mean_res[-1]):5.2f} gradi"
          f" | rms pull {np.sqrt(np.mean(pull**2)):.2f}")
slope = np.polyfit(np.log(N_VALUES), np.log(rms_err), 1)[0]
print(f"pendenza log-log dell'errore: {slope:+.2f} (attesa -0.5)")

plt.figure(figsize=(5.5, 4))
plt.loglog(N_VALUES, np.degrees(rms_err), "o-", label="rms errore")
plt.loglog(N_VALUES, np.degrees(mean_res), "s-", label="risoluzione dichiarata")
plt.loglog(N_VALUES, np.degrees(rms_err[0]) * np.sqrt(N_VALUES[0] / np.array(N_VALUES)), "k:", label=r"$1/\sqrt{N}$")
plt.xlabel("N"); plt.ylabel("gradi"); plt.legend(); plt.show()

Nota sul pull angolare: l'errore angolare è una distanza sulla sfera, quindi è non negativo e ha
due componenti. Se il posterior è circa gaussiano con larghezza $s$ per asse, $d^2 \approx s^2\chi^2_2$ e
$E[d^2] = 2s^2$. Il pull $d/\sqrt{E_\text{post}[d^2]}$ ha allora rms 1, e $\sqrt2\cdot$pull segue una
Rayleigh.

Lettura dei numeri: la pendenza è vicina a −0.5 e la risoluzione dichiarata segue l'errore reale.
L'rms del pull resta sotto 1 (0.7–0.9), cioè il posterior è conservativo. Il motivo è lo stesso del
generatore (ii) nel notebook 02: qui le energie vere sono $U(1, 5)$ MeV, più strette del prior
$[0.5, 6]$ MeV, e mancano proprio gli eventi ai bordi, dove il posterior sbaglia di più. Con M = 6
ripetizioni per N anche le fluttuazioni sono ampie. La checklist qui sotto estrae le energie dal prior
e dà rms vicino a 1.

## Checklist quantitativa del Caso B

`scripts/caso_B_checklist.py` ripete l'esperimento M volte per ogni N, con verità nuova a ogni
ripetizione: $\Omega_n$ isotropa e $E_n^{(k)}\sim U(0.5, 6)$ MeV, cioè dal prior largo che il Caso B usa
per marginalizzare. La cella legge i risultati salvati in `outputs/caso_B_checklist_results.pkl`
(rigenerabile con `OMP_NUM_THREADS=1 python scripts/caso_B_checklist.py 4`, circa 10 minuti).

**Esempio guidato: la costante della contrazione.** Se $\sigma_N = c/\sqrt N$, il prodotto
$\text{rms}\cdot\sqrt N$ deve essere costante. Con $c \approx 0.38$ rad (valore misurato a N = 100–300)
ci si aspetta $0.38/\sqrt{100} = 0.038$ rad $= 2.2^\circ$ a N = 100 e $0.38/\sqrt{1000} = 0.012$ rad
$= 0.69^\circ$ a N = 1000.

In [ ]:
import pickle
from pathlib import Path

with open(Path("..") / "outputs" / "caso_B_checklist_results.pkl", "rb") as f:
    B = pickle.load(f)
rms = lambda x: np.sqrt(np.mean(x ** 2))
n_values = sorted(n for n, _ in B["N_M"])
curve = []
for n in n_values:
    rows = [r for r in B["results"] if r["N"] == n]
    err = np.array([r["omega_error"] for r in rows]); sig = np.array([r["omega_sigma"] for r in rows])
    cov = np.mean([r["omega_covered"] for r in rows], axis=0)
    curve.append(rms(err))
    print(f"N={n:5d} M={len(rows):3d} | rms err {np.degrees(rms(err)):5.2f} gradi (dichiarata {np.degrees(sig.mean()):5.2f})"
          f" | pull rms {rms(validate.angular_pull(err, sig)):.2f} | coverage HPD {np.round(cov, 2)}"
          f" (+-{np.sqrt(0.68 * 0.32 / len(rows)):.2f})")
print(f"pendenza {np.polyfit(np.log(n_values), np.log(curve), 1)[0]:+.3f}   rms*sqrt(N) [rad] = "
      f"{np.round(np.array(curve) * np.sqrt(n_values), 3)}")

Lettura **(c)**:

* **bias e risoluzione**: la risoluzione dichiarata segue l'rms dell'errore reale a ogni N (per
  esempio 2.16° contro 2.15° a N = 100);
* **pull**: rms fra 0.99 e 1.03 per N = 10–300;
* **coverage**: nominale entro l'errore binomiale per N = 10–300;
* **contrazione**: $\text{rms}\cdot\sqrt N$ è stabile a circa 0.38 rad fra N = 100 e 300. A N = 10–30 è
  più alto perché il posterior di pochi eventi non è gaussiano, per cui la pendenza globale (−0.59) è
  più ripida di −0.5;
* **N = 1000**: con le 40 ripetizioni della checklist il pull rms era 0.88 e la coverage 0.78 al 68%
  (±0.07). Con 200 esperimenti aggiunti su seed disgiunti (`scripts/n1000_extra.py`, M = 240,
  cella seguente) si ottiene rms 0.65° contro 0.67° dichiarati, pull rms 0.96 e coverage HPD
  0.70 / 0.91 / 0.95 (±0.03): nominale. Lo 0.78 era una fluttuazione.

In [ ]:
import sys

sys.path.insert(0, "../scripts")
import n1000_extra

with open(Path("..") / "outputs" / "n1000_extra_results.pkl", "rb") as f:
    EXTRA = pickle.load(f)
rows_1000 = [r for r in B["results"] if r["N"] == 1000] + EXTRA["results"]["B"]
print("Caso B, N = 1000, checklist + esperimenti aggiunti:", n1000_extra.omega_summary(rows_1000))

## Il prior su $E_n$ non si dimentica, ma costa solo precisione

Nel Caso B $E_n$ è una nuisance **per evento**: ogni evento ha la sua, col suo prior. Aumentando N
il prior su $\Omega_n$ (parametro condiviso) si dimentica, come nel Caso A; quello su $E_n$ invece
no, perché ogni evento nuovo porta un'energia nuova da marginalizzare.

**Esempio guidato: due stimatori consistenti degli stessi dati.** Si misura N volte la stessa
quantità con errori $\sigma_k\sim U(0.5, 2)$ diversi per evento. Due stimatori:

* *default*: media pesata con $1/\sigma_k^2$, che è efficiente, con varianza
  $1/\sum_k\sigma_k^{-2}$;
* *alternativo*: media semplice, anch'essa senza bias, con varianza $\sum_k\sigma_k^2/N^2$.

La loro differenza $d$ è rumore a media nulla. Se il default è efficiente, $d$ è scorrelata da esso
e le varianze si sommano (relazione tipo Hausman) **(a)**:
$$E[d^2] = E[\text{err}_\text{alt}^2] - E[\text{err}_\text{def}^2].$$
Con $\sigma_k\sim U(0.5, 2)$ si ha $E[\sigma^{-2}] = \frac{1}{1.5}\int_{0.5}^{2}\sigma^{-2}d\sigma = 1$ e
$E[\sigma^2] = \frac{2^3 - 0.5^3}{3\cdot 1.5} = 1.75$. Quindi per N grande
$\text{var}_\text{def} = 1/N$, $\text{var}_\text{alt} = 1.75/N$, $E[d^2] = 0.75/N$ e
$$\text{rms}(d/\sigma_\text{def}) = \sqrt{0.75} = 0.87 \quad\text{per ogni } N.$$
Lo spostamento fra le due stime, in unità della risoluzione, **non scende con N**, eppure nessuna
delle due ha bias. È esattamente ciò che succede a $\hat\Omega_n$ cambiando il prior su $E_n$: il
prior sbagliato pesa male gli eventi.

In [ ]:
M_REP = 10_000
for n in (10, 100, 1000):
    sig_k = rng.uniform(0.5, 2.0, (M_REP, n))
    x = rng.normal(0.0, sig_k)
    w = 1.0 / sig_k**2
    est_def, s_def = (w * x).sum(axis=1) / w.sum(axis=1), 1.0 / np.sqrt(w.sum(axis=1))
    est_alt = x.mean(axis=1)
    d = est_alt - est_def
    print(f"N={n:5d} | E d^2 {np.mean(d**2):.5f}  vs  E alt^2 - E def^2 {np.mean(est_alt**2) - np.mean(est_def**2):.5f}"
          f" | corr(def, d) {np.corrcoef(est_def, d)[0, 1]:+.3f} | media d/s_def {np.mean(d / s_def):+.3f}"
          f" | rms d/s_def {np.sqrt(np.mean((d / s_def)**2)):.3f}")

Lettura **(c)**: la relazione regge a ogni N (per esempio 0.0074 contro 0.0074 a N = 100), la
correlazione fra stima di default e differenza è nulla e $\text{rms}(d/\sigma_\text{def})$ resta
vicino a 0.87. A N = 10 l'accordo è un po' peggiore perché $1/\sum\sigma_k^{-2}$ fluttua da un
esperimento all'altro.

Ora lo stesso per il Caso B (`scripts/robustezza_prior.py`, Caso B, stessi esperimenti per ogni
prior). Prior alternativi, tutti propri: $E_n$ piatto su [0.5, 10] MeV ("piatto largo"),
log-uniforme su [0.5, 6] MeV, e un prior von Mises–Fisher su $\Omega_n$ con $\kappa = 2$ (largo
circa 40°) e asse a 90° dalla verità. La metrica è l'angolo fra la stima alternativa e quella di
default, in unità della risoluzione del default. Poi `scripts/prior_B_rumore.py` controlla, sugli
stessi esperimenti, se lo spostamento è rumore o bias e se vale la relazione tipo Hausman.

In [ ]:
import robustezza_prior
import prior_B_rumore

with open(Path("..") / "outputs" / "robustezza_prior_results.pkl", "rb") as f:
    ROB = pickle.load(f)
robustezza_prior.print_direction_table(ROB["results"]["B"], ["piatto", "piatto largo", "logU", "vMF"])
print()
prior_B_rumore.main()

Lettura:

* **prior su $\Omega_n$ (vMF a 90°)**: lo spostamento scende come atteso, 0.21 / 0.09 / 0.04 /
  0.02 $\sigma$ per N = 10 / 30 / 100 / 300 **(c)**;
* **prior su $E_n$**: lo spostamento resta circa 1 $\sigma$ col piatto largo (1.12 / 1.18 / 1.04 /
  1.22) e circa 0.3 $\sigma$ col log-uniforme (0.46 / 0.34 / 0.32 / 0.29) **(c)**. Col piatto largo
  l'errore rms cresce (17.0° contro 9.8° a N = 10, 2.0° contro 1.3° a N = 300), ma la risoluzione
  dichiarata cresce con esso: pull rms 0.90–0.99 e coverage nominale o sopra;
* **meccanismo**: il vettore di spostamento nel piano tangente ha media compatibile con zero, cioè
  è rumore e non bias. Col piatto largo da N = 30 vale la relazione tipo Hausman, e il rapporto
  rms(err_alt)/rms(err_def) è circa costante con N (1.74 / 1.75 / 1.60 / 1.58): ogni evento spreca
  la stessa frazione di informazione, come nell'esempio guidato **(c)**;
* **eccezioni**: a N = 10 col piatto largo la correlazione err_def·d è +0.26 (il posterior non è
  gaussiano e il MAP non è la media); il log-uniforme a N ≤ 30 non è meno efficiente del default
  (rapporto 0.98–1.03), quindi lo spostamento di 0.3 $\sigma$ è rumore fra due stime equivalenti.

In pratica il prior su $E_n$ va scelto sullo spettro fisico atteso: un prior sbagliato costa
precisione, non calibrazione **(c)**.

## Stress dell'assunzione 1: una seconda sorgente

Ora una frazione $f$ degli eventi viene da una seconda sorgente puntiforme a separazione $\Delta$
(10°, 30°, 90°) o da un fondo isotropo, mentre la ricostruzione resta a sorgente unica
(`scripts/stress_sorgente.py`).

**Esempio guidato.** Se la stima si comporta come una media pesata delle direzioni, si sposta di
circa $f\Delta$ verso la seconda sorgente: con $f = 0.1$ e $\Delta = 10°$ è 1°. A N = 300 la
risoluzione è circa 1.2°, quindi il bias vale già quasi 1 $\sigma$. La risoluzione scende come
$1/\sqrt N$ mentre il bias resta fisso: il pull cresce come $\sqrt N$ e **più eventi peggiorano
la coverage**.

La diagnostica è il test KS del notebook 01 sugli angoli fra le tracce e $\hat\Omega_n$. Il test
ha potenza solo se la distanza fra la distribuzione contaminata e quella a sorgente unica,
$D_\infty$, è confrontabile con la soglia $\approx 1.36/\sqrt N$ = 0.079 a N = 300. La cella
seguente calcola $D_\infty$ con molte tracce e la verità nota.

In [ ]:
n_big = 400_000
omega0 = OMEGA_TRUE[0]
obs_grid, cdf = forward_model.track_angle_cdf(SIGMA_THETA)
for frac in (0.05, 0.10, 0.30):
    row = []
    for sep in (10.0, 30.0, 90.0, None):
        axes_k = kinematics.mixed_source_axes(rng, omega0, n_big, frac, None if sep is None else np.radians(sep))
        _, tr = kinematics.sample_recoil_events(rng, rng.uniform(1.0, 5.0, n_big), axes_k)
        th = kinematics.recoil_angle_from_direction(kinematics.smear_direction(rng, tr, SIGMA_THETA), omega0[None, :])[:, 0]
        row.append(validate.ks_uniform_statistic(validate.probability_integral_transform(th, obs_grid, cdf)))
    print(f"f={frac:.2f}: D_inf per Delta = 10 / 30 / 90 gradi / isotropo: {np.round(row, 4)}"
          f"   sqrt(300)*D_inf: {np.round(np.sqrt(300) * np.array(row), 2)}")
print(f"soglia a N=300: 1.36/sqrt(300) = {1.36 / np.sqrt(300):.4f}")

In [ ]:
with open(Path("..") / "outputs" / "stress_sorgente_results.pkl", "rb") as f:
    SRC = pickle.load(f)


def source_table(case: str, n: int) -> None:
    sel_n = [r for r in SRC["results"] if r["case"] == case and r["N"] == n]
    thr = np.quantile([r["ks"] for r in sel_n if r["f"] == 0], 0.95)
    print(f"Caso {case}, N = {n}; soglia KS (95 percentile a f=0) = {thr:.4f}")
    configs = [(0.0, None)] + [(f, s) for s in SRC["SEPARATIONS_DEG"] for f in SRC["FRACTIONS"]]
    for frac, sep in configs:
        sel = [r for r in sel_n if r["f"] == frac and r["sep"] == sep]
        err = np.array([r["omega_error"] for r in sel]); sig = np.array([r["omega_sigma"] for r in sel])
        toward = np.array([r["toward_second"] for r in sel])
        label = "nessuna" if frac == 0 else f"f={frac:.2f} " + ("isotropo" if sep is None else f"{sep:.0f} gradi")
        shift = f"{np.degrees(np.nanmean(toward)):5.2f}" if np.isfinite(toward).any() else "   - "
        line = (f"  {label:20s} M={len(sel):3d} | verso la 2a {shift} gradi | pull rms {rms(validate.angular_pull(err, sig)):5.2f}"
                f" | cov68 {np.mean([r['omega_covered'][0] for r in sel]):.2f} | potenza KS {np.mean([r['ks'] > thr for r in sel]):.2f}")
        if case == "C":
            for key in ("mu", "ls"):
                pull = np.array([(r[key]["mean"] - r[f"{key}_true"]) / r[key]["std"] for r in sel])
                line += f" | pull {key} {pull.mean():+.2f}"
        print(line)


for n in SRC["N_VALUES"]["B"]:
    source_table("B", n); print()

Lettura:

* $D_\infty$ è sotto 0.012 per $\Delta \le 30°$ e $f \le 0.1$, ben sotto la soglia: lì il test **non
  può** vedere la contaminazione, a nessun N realistico. A 90° con $f = 0.1$ si ha
  $D_\infty = 0.060$, confrontabile con la soglia, e la potenza osservata è 0.80 **(c)**;
* **lo spostamento** è circa $f\Delta$ per $\Delta = 10°$ (0.91° a $f = 0.1$), più grande a 30° (4.0°
  contro 3°) e molto più grande a 90° (16.9° contro 9°) **(c)**; la lettura come media pesata è
  **(d)**;
* **la coverage crolla** mentre il test tace: a N = 300 con $f = 0.05$ a 30° la coverage 68% è
  0.16, e la potenza KS resta al livello del falso allarme. A N = 30 la stessa configurazione dà
  0.41: più eventi, peggio **(c)**.

L'assunzione di sorgente unica resta quindi **necessaria e non verificabile** con questa
diagnostica per separazioni ≲ 30°. Un modello a mistura che stimi $f$ è il passo naturale
successivo, fuori scope.